# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides an example workflow for exploring the FAIR^2 dataset ["Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells"](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
This dataset is accessible via a Croissant schema URL and is designed for machine learning and computational analysis using the Croissant standard.

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install mlcroissant

## 1. Data Loading

We'll load the dataset metadata and then review its top-level attributes using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)

print(f"Dataset Title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}\n")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview

Now, we review the available record sets and their corresponding fields, listing all IDs. All Croissant entities (such as record sets and fields) have a unique `@id` which should be used for reference and extraction to ensure consistency.

Let's enumerate the record sets with their IDs and, for each, display its available fields (i.e., columns).

In [ ]:
# List available record sets, fields and their `@id`s
record_sets = list(dataset.metadata.record_sets)

print(f"There are {len(record_sets)} record sets in this dataset.\n")
rs_to_fields = {}
for rs in record_sets:
    print(f"RecordSet: id={rs['@id']} name={rs['name'] if 'name' in rs else 'N/A'}")
    # List fields (columns) for this record set, if present
    if 'fields' in rs:
        print("  Fields (by @id):")
        for field in rs['fields']:
            print(f"    - {field['@id']} name={field.get('name', 'N/A')} type={field.get('dataType', 'N/A')}")
        rs_to_fields[rs['@id']] = [f["@id"] for f in rs['fields']]
    elif 'columns' in rs:
        print("  Columns (by @id):")
        for col in rs['columns']:
            print(f"    - {col['@id']} name={col.get('name', 'N/A')} type={col.get('dataType', 'N/A')}")
        rs_to_fields[rs['@id']] = [f["@id"] for f in rs['columns']]
    else:
        print("  No fields/columns listed.")
    print()
if not record_sets:
    print("No record sets declared in top-level schema. If so, please check the online description or inspect the full metadata object.")

## 3. Data Extraction

Let's extract data from a specific record set using its `@id`.

We'll demonstrate with the first available RecordSet by `@id`, extracting its records into a DataFrame for analysis.

In [ ]:
# Try extracting data from each available recordSet
import warnings
warnings.filterwarnings('ignore')

dataframes = {}
if record_sets:
    for rs in record_sets:
        rs_id = rs['@id']
        print(f"\nExtracting {rs_id} ...")
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"  - Loaded {len(df)} records.")
            print(f"  - Columns: {df.columns.tolist()}")
            display(df.head())
        except Exception as e:
            print(f"  - Failed to extract data: {e}")

if not dataframes:
    print("No structured record sets could be loaded. Consult the Croissant metadata for manual inspection.")

## 4. Exploratory Data Analysis (EDA)

We'll apply basic EDA on a chosen record set and numeric field, demonstrating:
- Filtering records
- Normalizing a numeric field
- Grouping by a key attribute

Remember to use the column `@id` exactly as shown in the Data Overview above.

In [ ]:
# Parameters (edit to fit available IDs from previous exploration)

# For demonstration, we select the first (if any) loaded record set and try to process its first numeric field
filtered_df = None
chosen_rs_id = None
numeric_field_id = None

for rs_id, df in dataframes.items():
    for col in df.columns:
        # Heuristically pick the first float/integer column
        if pd.api.types.is_numeric_dtype(df[col]):
            chosen_rs_id = rs_id
            numeric_field_id = col
            break
    if chosen_rs_id:
        break

if chosen_rs_id and numeric_field_id:
    print(f"Analyzing numeric field '{numeric_field_id}' in record set '{chosen_rs_id}'\n")
    # Example: filter on a threshold
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: (showing top 5)")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by another column (first non-numeric)
    group_field = None
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]) and col != numeric_field_id:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
        grouped_df.columns = [f"mean_{numeric_field_id}"]
        print(f"Grouped mean of {numeric_field_id} by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable categorical/text field found for grouping.")
else:
    print("No numeric field found for analysis in available data frames.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field using a histogram and a boxplot. For grouped data, we can show a bar plot by categories (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_rs_id and numeric_field_id and filtered_df is not None and not filtered_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(filtered_df[numeric_field_id], ax=axes[0], kde=True)
    axes[0].set_title(f"Distribution of {numeric_field_id}")
    sns.boxplot(y=filtered_df[numeric_field_id], ax=axes[1])
    axes[1].set_title(f"Boxplot of {numeric_field_id}")
    plt.tight_layout()
    plt.show()
    # Show barplot if grouped data is available
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(8,4))
        grouped_df.plot(kind='bar', ax=plt.gca(), legend=False)
        plt.title(f"Grouped mean of {numeric_field_id} by {group_field}")
        plt.ylabel(f"mean_{numeric_field_id}")
        plt.xlabel(f"{group_field}")
        plt.tight_layout()
        plt.show()
else:
    print("No data available to plot. Check earlier cells for issues with field selection.")

## 6. Conclusion

- We used `mlcroissant` to inspect, extract, and explore the FAIR^2 dataset package for mass spectrometry data analysis.
- By referring strictly to each entity's `@id`, downstream steps remain robust and reproducible.
- Further analysis can be done by exploring additional record sets or by integrating Croissant with compatible machine learning workflows.

For more examples or advanced usage of the Croissant standard, visit [https://mlcommons.github.io/croissant/](https://mlcommons.github.io/croissant/).
